# mart_dev.job_run_report

A Saved table the batch job_run_report made, with project.convert(...), from the statements in `conversion/statement.hql`, as its script wrote them. Its logic cell builds the same query from the Toolbox's own functions, with create(), preview() and write(). Answer its open Decisions, which `conversion/checklist.md` lists, in the decisions cell of the convert Workflow's notebook, `admin/workflows/convert/convert.ipynb`, then run the cells top to bottom. The cells comparing it with the original stay until you delete them.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project: two up

from sqlglot_composer import (compare_hive, compare_with_original, export_hive, export_lineage,
                              project, run_query, show_hive)
from warehouse import query

In [ ]:
# logic
from sqlglot_composer import (
    AS,
    Clause,
    FROM,
    GROUP_BY,
    SELECT,
    Statement,
    average_of,
    count_rows,
    equals,
    fill_null,
    statement,
)
from references import ops
from sqlglot_composer import create_table  # for create()
from sqlglot_composer import INSERT_OVERWRITE, Statement, statement  # for write()
from references.mart_dev__job_run_report import job_run_report  # the Saved table's own Table reference

job_runs = ops.job_runs


def rows() -> list[Clause]:
    """The clauses of the query, which converted() and a write share."""
    return [
        SELECT(
            job_runs.job_id,
            AS(fill_null(count_rows(), 0), "runs"),
            AS(fill_null(count_rows(where=equals(job_runs.status, "FAILED")), 0), "failed_runs"),
            AS(average_of(job_runs.duration_mins), "avg_duration_mins"),
        ),
        # the workbook counts every row of every day, so this reads every partition
        FROM(job_runs, reads_all_partitions=True),
        GROUP_BY(job_runs.job_id),
    ]


def converted() -> Statement:
    """The query, as the Hive this Conversion was made from writes it."""
    return statement(*rows())


def create() -> Statement:
    """Create mart_dev.job_run_report from its Table reference, if it isn't there yet."""
    return create_table(job_run_report, may_exist=True)


def preview() -> Statement:
    """The rows write() would save, as a SELECT to run and check first."""
    return converted()


def write() -> Statement:
    """Save the rows into mart_dev.job_run_report, as the original's INSERT did."""
    return statement(INSERT_OVERWRITE(job_run_report), *rows())


In [ ]:
# Try it: its rows, then, once its Table reference has its location=, the Hive of
# each step
run_query(preview(), query)
show_hive([create(), write()])

In [ ]:
# Lineage: a new pair in lineage/, named by the time and this table
export_lineage(write())

In [ ]:
# Its Hive: mart_dev__job_run_report.hql beside this notebook, from the logic as last saved
export_hive()

In [ ]:
# Its Hive against the original's, as conversion/in_dev.hql holds it
ORIGINAL = pathlib.Path("conversion/in_dev.hql").read_text(encoding="utf-8")
show_hive(preview())
print(compare_hive(preview(), ORIGINAL))

In [ ]:
# The convert Workflow's step 6, project.compare_batch, compares this table once written.
# This compares its query with the original's first:
# Its rows against the original's, through your query, on one Partition:
# print(compare_with_original(preview(), ORIGINAL, query=query, partition={"dt": "<a value>"}))

In [ ]:
# Its rows in the Reference sheet
project.refresh_reference_sheet("mart_dev.job_run_report")